In [1]:
# Colab cell — Paste into Google Colab and run.
# Upload a .zip containing one or more CSVs (your Zeek connection logs),
# it will extract, read, concatenate, show first/last 5 rows and save merged CSV.

from google.colab import files
import zipfile, os, glob
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

print("Step 1/4 — Upload a .zip file containing your CSV(s).")
uploaded = files.upload()  # prompts upload dialog in Colab

if not uploaded:
    raise SystemExit("No files uploaded. Rerun the cell and upload a .zip file.")

zip_filename = list(uploaded.keys())[0]
print(f"Uploaded: {zip_filename}")

extract_dir = "extracted_csvs"
os.makedirs(extract_dir, exist_ok=True)

print("Step 2/4 — Extracting CSV files from the zip...")
with zipfile.ZipFile(zip_filename, "r") as z:
    z.extractall(path=extract_dir)
    csv_list = [f for f in z.namelist() if f.lower().endswith(".csv")]

# If namelist entries were folders, fall back to glob
if not csv_list:
    csv_list = glob.glob(os.path.join(extract_dir, "*.csv"))

if not csv_list:
    raise SystemExit("No CSV files found inside the uploaded zip. Please include CSV(s) and try again.")

print(f"Found {len(csv_list)} CSV file(s):")
for i,f in enumerate(csv_list,1):
    print(f"  {i}. {f}")

print("Step 3/4 — Reading CSV files and concatenating (this may take a while for large files)...")
dfs = []
for f in csv_list:
    path = os.path.join(extract_dir, os.path.basename(f))
    try:
        df = pd.read_csv(path, low_memory=False)
    except Exception as e:
        print(f"Failed to read {path} with pd.read_csv(): {e}\nTrying engine='python'...")
        try:
            df = pd.read_csv(path, engine='python')
        except Exception as e2:
            print(f"Failed to read {path} with engine='python': {e2}\nSkipping this file.")
            continue
    print(f"  Loaded {os.path.basename(path)} -> shape {df.shape}")
    dfs.append(df)

if not dfs:
    raise SystemExit("No CSVs could be read. Exiting.")

merged_df = pd.concat(dfs, ignore_index=True, copy=False)
print(f"Step 4/4 — Merged DataFrame shape: {merged_df.shape}")

print("\n--- Columns and dtypes ---")
for col, dtype in merged_df.dtypes.items():
    print(f"{col:40} : {str(dtype)}")

print("\n--- 5 first rows ---")
display(merged_df.head(5))

print("\n--- 5 last rows ---")
display(merged_df.tail(5))

# Quick helpful checks
print("\nQuick stats:")
print(f"  Total rows: {len(merged_df)}")
print(f"  Total columns: {len(merged_df.columns)}")
print("Top 10 destination ports (if present):")
if "destination.port" in merged_df.columns:
    print(merged_df["destination.port"].value_counts().head(10))
else:
    for alt in ["dest_port", "dst_port", "dport", "destination_port"]:
        if alt in merged_df.columns:
            print(merged_df[alt].value_counts().head(10))
            break
    else:
        print("  (destination port column not found)")

# Save merged CSV for later use
out_fn = "merged_zeek_connection.csv"
merged_df.to_csv(out_fn, index=False)
print(f"\nMerged CSV saved to {out_fn}. To download it in Colab use:\nfrom google.colab import files\nfiles.download('{out_fn}')")


Step 1/4 — Upload a .zip file containing your CSV(s).


Saving merged.conn.new.with.ip.zip to merged.conn.new.with.ip.zip
Uploaded: merged.conn.new.with.ip.zip
Step 2/4 — Extracting CSV files from the zip...
Found 1 CSV file(s):
  1. merged.conn.new.with.ip.csv
Step 3/4 — Reading CSV files and concatenating (this may take a while for large files)...
  Loaded merged.conn.new.with.ip.csv -> shape (2169590, 21)
Step 4/4 — Merged DataFrame shape: (2169590, 21)

--- Columns and dtypes ---
@timestamp                               : object
event.duration                           : object
network.bytes                            : object
network.packets                          : object
source.bytes                             : object
source.packets                           : object
source.port                              : object
source.ip                                : object
destination.bytes                        : object
destination.packets                      : object
destination.port                         : object
destination.ip   

,@timestamp,event.duration,network.bytes,network.packets,source.bytes,source.packets,source.port,source.ip,destination.bytes,destination.packets,...,destination.ip,zeek.connection.missed_bytes,network.transport,network.protocol,zeek.connection.state,zeek.connection.history,network.direction,zeek.connection.local_orig,zeek.connection.local_resp,zeek.connection.icmp.type
0,"Oct 16, 2025 @ 21:59:59.864","315,993,071","5,982",110,"2,664",42,"49,538",192.168.20.15,"3,318",68,...,172.16.60.2,0,tcp,http,SF,ShADTadttfF,internal,True,True,-
1,"Oct 16, 2025 @ 21:59:59.841","325,828,075","5,650",108,"2,332",40,"49,537",192.168.20.15,"3,318",68,...,172.16.60.2,0,tcp,http,SF,ShADTadttfF,internal,True,True,-
2,"Oct 16, 2025 @ 21:59:59.828","3,088,467,836","39,476",68,"30,392",42,"58,404",192.168.100.5,"9,084",26,...,192.168.100.112,0,tcp,ssl,SF,ShADTadtTFfR,internal,True,True,-
3,"Oct 16, 2025 @ 21:59:59.818","321,600,914","6,078",110,"2,760",42,"49,536",192.168.20.15,"3,318",68,...,172.16.60.2,0,tcp,http,SF,ShADTadttfF,internal,True,True,-
4,"Oct 16, 2025 @ 21:59:59.598","224,829",184,4,104,2,"10,253",192.168.100.14,80,2,...,74.178.76.128,0,tcp,-,REJ,Sr,outbound,True,False,-



--- 5 last rows ---


,@timestamp,event.duration,network.bytes,network.packets,source.bytes,source.packets,source.port,source.ip,destination.bytes,destination.packets,...,destination.ip,zeek.connection.missed_bytes,network.transport,network.protocol,zeek.connection.state,zeek.connection.history,network.direction,zeek.connection.local_orig,zeek.connection.local_resp,zeek.connection.icmp.type
2169585,"Nov 10, 2025 @ 08:33:19.250","36,992,073",184,4,104,2,"9,314",10.10.10.31,80,2,...,10.10.34.35,0,tcp,-,REJ,Sr,internal,True,True,-
2169586,"Nov 10, 2025 @ 08:33:19.207","27,898,073",184,4,104,2,"1,143",10.10.10.31,80,2,...,10.10.34.35,0,tcp,-,REJ,Sr,internal,True,True,-
2169587,"Nov 10, 2025 @ 08:33:18.991","3,300,031,185","19,392",56,"19,392",56,"1,900",192.168.100.251,0,0,...,239.255.255.250,0,udp,-,S0,D,outbound,True,False,-
2169588,"Nov 10, 2025 @ 08:33:18.928","3,007,045,984",312,6,312,6,"3,149",10.10.10.31,0,0,...,10.10.34.35,0,tcp,-,S0,S,internal,True,True,-
2169589,"Nov 10, 2025 @ 08:33:18.769","128,031",184,4,104,2,"57,891",192.168.20.15,80,2,...,173.243.143.6,0,tcp,-,REJ,Sr,outbound,True,False,-



Quick stats:
  Total rows: 2169590
  Total columns: 21
Top 10 destination ports (if present):
destination.port
443       682637
8,080     550241
53        390731
80        141414
9,200      71717
902        50272
21,116     45868
9,080      39394
5,060      37807
8,383      16053
Name: count, dtype: int64

Merged CSV saved to merged_zeek_connection.csv. To download it in Colab use:
from google.colab import files
files.download('merged_zeek_connection.csv')


In [2]:
# =======================
# 🔹 Cell 2 — Clean & Prepare DataFrame for row-level + 1-min aggregation
#     - Fix dtypes (remove thousands separators like "8,080")
#     - Parse timestamps
#     - Convert event.duration (ns → seconds)
#     - Create basic row-level features (bps, pps, bpp, byte ratios)
#     - Produce df_clean for next steps
# =======================
import re
import numpy as np
import pandas as pd

# If you still have `merged_df` from Cell 1, we use it. Otherwise read from disk:
try:
    df = merged_df.copy()
except NameError:
    df = pd.read_csv("merged_zeek_connection.csv", low_memory=False)

print(f"Input shape: {df.shape}")

# -------- Helpers --------
def to_numeric_clean(series, allow_float=False):
    """
    Remove commas and non-digit separators; convert to Int64 (nullable) or float.
    Accepts values like '8,080' or '-' (becomes NaN).
    """
    # keep digits, optional decimal dot, and minus sign
    s = series.astype(str).str.replace(",", "", regex=False).str.strip()
    s = s.replace({"-": np.nan, "None": np.nan, "null": np.nan, "NaN": np.nan})
    if allow_float:
        return pd.to_numeric(s, errors="coerce")
    else:
        # Integers: coerce floats like '0.0' to int later
        out = pd.to_numeric(s, errors="coerce")
        # If there are floats but intended integer, floor them safely
        if pd.api.types.is_float_dtype(out):
            # Keep as pandas nullable integer when possible
            if (out.dropna() % 1 == 0).all():
                out = out.astype("Int64")
        return out

def safe_div(a, b, eps=1e-9):
    return a / (b + eps)

# -------- 1) Parse timestamp --------
# Example format: "Oct 16, 2025 @ 21:59:59.864"
ts = df["@timestamp"].astype(str).str.replace(" @ ", " ", regex=False)
df["ts"] = pd.to_datetime(ts, errors="coerce")  # tz-naive; keep as-is

# -------- 2) Fix numeric columns (remove thousands separators and cast) --------
int_like_cols = [
    "event.duration",
    "network.bytes", "network.packets",
    "source.bytes", "source.packets", "source.port",
    "destination.bytes", "destination.packets", "destination.port",
    "zeek.connection.missed_bytes",
    "zeek.connection.icmp.type",
]
for c in int_like_cols:
    if c in df.columns:
        # duration may overflow int64 as string, keep as numeric (float) then cast
        allow_float = (c == "event.duration")
        df[c] = to_numeric_clean(df[c], allow_float=allow_float)

# Optional: transport/protocol to category
for c in ["network.transport", "network.protocol", "network.direction",
          "zeek.connection.state", "zeek.connection.history"]:
    if c in df.columns:
        df[c] = df[c].astype("category")

# -------- 3) Duration in seconds (assuming ns) --------
# Zeek's event.duration from Elastic is commonly in nanoseconds.
if "event.duration" in df.columns:
    df["duration_s"] = df["event.duration"] / 1e9
else:
    df["duration_s"] = np.nan

# Guard for zero/neg durations for rate computations
df["duration_s_clipped"] = df["duration_s"].clip(lower=1e-6)

# -------- 4) Basic row-level features --------
# bpp: bytes per packet, pps: packets per second, bps: bytes per second
if {"network.bytes", "network.packets"}.issubset(df.columns):
    df["bpp"] = safe_div(df["network.bytes"], df["network.packets"].replace(0, np.nan)).fillna(0.0)

df["pps"] = safe_div(df.get("network.packets", 0), df["duration_s_clipped"])
df["bps"] = safe_div(df.get("network.bytes", 0), df["duration_s_clipped"])

# Byte asymmetry ratios
if {"source.bytes", "destination.bytes"}.issubset(df.columns):
    df["src_to_dst_byte_ratio"] = safe_div(df["source.bytes"] + 1, df["destination.bytes"] + 1)
    df["dst_to_src_byte_ratio"] = safe_div(df["destination.bytes"] + 1, df["source.bytes"] + 1)

# Packet asymmetry ratios
if {"source.packets", "destination.packets"}.issubset(df.columns):
    df["src_to_dst_pkt_ratio"] = safe_div(df["source.packets"] + 1, df["destination.packets"] + 1)
    df["dst_to_src_pkt_ratio"] = safe_div(df["destination.packets"] + 1, df["source.packets"] + 1)

# Port categorization (well-known 0–1023, registered 1024–49151, dynamic 49152–65535)
def port_bucket(s):
    s = s.astype("Int64")
    out = pd.Series(pd.Categorical(["unknown"] * len(s), categories=["well_known","registered","dynamic","unknown"]))
    mask = s.notna()
    out.loc[mask & (s.between(0, 1023))] = "well_known"
    out.loc[mask & (s.between(1024, 49151))] = "registered"
    out.loc[mask & (s.between(49152, 65535))] = "dynamic"
    return out.astype("category")

if "destination.port" in df.columns:
    df["dst_port_bucket"] = port_bucket(df["destination.port"])
if "source.port" in df.columns:
    df["src_port_bucket"] = port_bucket(df["source.port"])

# -------- 5) Final tidy DataFrame --------
# Keep original columns + engineered features; sort by timestamp
df_clean = df.sort_values("ts").reset_index(drop=True)

# -------- 6) Quick sanity outputs --------
print("\n=== After cleaning ===")
print("Shape:", df_clean.shape)
print("\nDtypes summary:")
print(df_clean.dtypes)

print("\nTime range (ts):", df_clean["ts"].min(), "→", df_clean["ts"].max())

# Show a compact preview of key columns
preview_cols = [
    "ts", "duration_s", "network.bytes", "network.packets",
    "source.ip", "source.port", "destination.ip", "destination.port",
    "bps", "pps", "bpp", "src_to_dst_byte_ratio",
    "network.transport", "network.protocol", "zeek.connection.state",
    "zeek.connection.history", "network.direction"
]
preview = [c for c in preview_cols if c in df_clean.columns]
print("\n--- Head(5) of cleaned selection ---")
display(df_clean[preview].head(5))

print("\n--- Tail(5) of cleaned selection ---")
display(df_clean[preview].tail(5))

# Some quick distributions for sanity
if "destination.port" in df_clean.columns:
    print("\nTop 15 destination ports after cleaning:")
    print(df_clean["destination.port"].value_counts(dropna=False).head(15))

if "zeek.connection.state" in df_clean.columns:
    print("\nState distribution:")
    print(df_clean["zeek.connection.state"].value_counts(dropna=False))

if "zeek.connection.missed_bytes" in df_clean.columns:
    mb = df_clean["zeek.connection.missed_bytes"]
    print(f"\nmissed_bytes — nonzero rows: {(mb.fillna(0) > 0).sum()} / {len(mb)}")

print("\n✅ df_clean is ready for:")
print("   (a) row-level modeling (IF/AE)")
print("   (b) 1-minute aggregations (next cell)")


Input shape: (2169590, 21)

=== After cleaning ===
Shape: (2169590, 33)

Dtypes summary:
@timestamp                              object
event.duration                         float64
network.bytes                            int64
network.packets                          int64
source.bytes                             int64
source.packets                           int64
source.port                              Int64
source.ip                               object
destination.bytes                        int64
destination.packets                      int64
destination.port                         Int64
destination.ip                          object
zeek.connection.missed_bytes             int64
network.transport                     category
network.protocol                      category
zeek.connection.state                 category
zeek.connection.history               category
network.direction                     category
zeek.connection.local_orig                bool
zeek.connection.lo

,ts,duration_s,network.bytes,network.packets,source.ip,source.port,destination.ip,destination.port,bps,pps,bpp,src_to_dst_byte_ratio,network.transport,network.protocol,zeek.connection.state,zeek.connection.history,network.direction
0,2025-10-16 20:13:49.580,0.000054,184,4,192.168.100.14,7704,146.75.122.172,80,3.414743e+06,74233.538713,46.0,1.296296,tcp,-,REJ,Sr,outbound
1,2025-10-16 20:13:49.681,0.000051,184,4,192.168.100.5,55590,20.44.239.154,443,3.606217e+06,78396.017482,46.0,1.296296,tcp,-,REJ,Sr,outbound
2,2025-10-16 20:13:49.691,0.000110,184,4,192.168.100.5,55592,20.44.239.154,443,1.670449e+06,36314.117113,46.0,1.296296,tcp,-,REJ,Sr,outbound
3,2025-10-16 20:13:49.704,0.000489,378,4,192.168.100.31,33401,192.168.100.5,53,7.730093e+05,8179.992556,94.5,0.576763,udp,dns,SF,Dd,internal
4,2025-10-16 20:13:49.704,0.000479,308,4,192.168.100.31,50220,192.168.100.5,53,6.430277e+05,8351.009637,77.0,0.812865,udp,dns,SF,Dd,internal



--- Tail(5) of cleaned selection ---


,ts,duration_s,network.bytes,network.packets,source.ip,source.port,destination.ip,destination.port,bps,pps,bpp,src_to_dst_byte_ratio,network.transport,network.protocol,zeek.connection.state,zeek.connection.history,network.direction
2169585,2025-11-10 11:34:45.475,0.000005,104,2,192.168.20.13,56119,192.168.100.33,8383,2.076677e+07,399361.022364,52.0,105.000000,tcp,-,S0,S,internal
2169586,2025-11-10 11:34:45.646,0.000006,104,2,192.168.20.13,56120,192.168.100.33,8383,1.744674e+07,335514.175474,52.0,105.000000,tcp,-,S0,S,internal
2169587,2025-11-10 11:34:45.848,0.000006,104,2,192.168.20.13,56121,192.168.100.33,8383,1.744674e+07,335514.175474,52.0,105.000000,tcp,-,S0,S,internal
2169588,2025-11-10 11:34:45.999,0.000007,104,2,192.168.20.13,56122,192.168.100.33,8383,1.503977e+07,289226.319595,52.0,105.000000,tcp,-,S0,S,internal
2169589,2025-11-10 11:34:48.426,0.000443,184,4,192.168.20.15,64927,37.32.10.75,21116,4.153658e+05,9029.691884,46.0,1.296296,tcp,-,REJ,Sr,outbound



Top 15 destination ports after cleaning:
destination.port
443      682637
8080     550241
53       390731
80       141414
9200      71717
902       50272
21116     45868
9080      39394
5060      37807
8383      16053
6568      14500
123       12933
1514      11891
9741       8927
21114      7900
Name: count, dtype: Int64

State distribution:
zeek.connection.state
SF        1107902
REJ        719821
S0         241878
RSTO        61453
RSTR        15169
OTH          6568
S1           4897
RSTRH        3543
SHR          3428
SH           1751
S3           1670
S2           1463
RSTOS0         47
Name: count, dtype: int64

missed_bytes — nonzero rows: 51185 / 2169590

✅ df_clean is ready for:
   (a) row-level modeling (IF/AE)
   (b) 1-minute aggregations (next cell)


In [3]:
# =======================
# 🔹 Cell 3 — 1-minute Aggregation Features + Show first/last 5 rows
# =======================
import numpy as np
import pandas as pd

print("Starting 1-minute aggregation... (this may take a bit)")

df_tmp = df_clean.copy()
df_tmp = df_tmp.set_index("ts").sort_index()

# ستون‌های عددی کلیدی
num_cols = [
    "network.bytes", "network.packets",
    "duration_s", "bps", "pps", "bpp",
    "src_to_dst_byte_ratio", "dst_to_src_byte_ratio",
    "src_to_dst_pkt_ratio", "dst_to_src_pkt_ratio",
    "zeek.connection.missed_bytes"
]
num_cols = [c for c in num_cols if c in df_tmp.columns]

# توابع آماری برای هر ستون عددی
agg_funcs = {c: ["count", "sum", "mean", "std", "max", "min"] for c in num_cols}

# اجرای resample برای بازه‌های 1 دقیقه‌ای
agg_minute = df_tmp.resample("1min").agg(agg_funcs)
agg_minute.columns = [f"{c[0]}_{c[1]}" for c in agg_minute.columns]
agg_minute = agg_minute.reset_index()

# fan-out و fan-in ساده
agg_fanout = (
    df_tmp.groupby(pd.Grouper(freq="1min"))
    .agg({
        "source.ip": pd.Series.nunique,
        "destination.ip": pd.Series.nunique,
        "destination.port": pd.Series.nunique
    })
    .rename(columns={
        "source.ip": "unique_src_ips",
        "destination.ip": "unique_dst_ips",
        "destination.port": "unique_dst_ports"
    })
    .reset_index()
)

# ادغام نتایج
df_agg_1min = pd.merge(agg_minute, agg_fanout, on="ts", how="left")

# نرخ‌های کمکی
df_agg_1min["bytes_per_flow_mean"] = (df_agg_1min["network.bytes_sum"] /
                                      df_agg_1min["network.bytes_count"].replace(0, np.nan))
df_agg_1min["packets_per_flow_mean"] = (df_agg_1min["network.packets_sum"] /
                                        df_agg_1min["network.packets_count"].replace(0, np.nan))

# پر کردن NaN با صفر برای مدل‌ها
df_agg_1min = df_agg_1min.fillna(0)

print("\n✅ Aggregation complete.")
print("Shape:", df_agg_1min.shape)
print("Time range:", df_agg_1min['ts'].min(), "→", df_agg_1min['ts'].max())

# نمایش نمونه‌ها
print("\n--- 5 first rows ---")
display(df_agg_1min.head(5))

print("\n--- 5 last rows ---")
display(df_agg_1min.tail(5))

# چک سریع فیچرها
print("\nColumns in aggregated dataset:")
print(df_agg_1min.columns.tolist())

# ذخیره برای مرحله‌ی مدل‌سازی
df_agg_1min.to_csv("agg_1min_features.csv", index=False)
print("\nSaved 1-minute aggregated features to agg_1min_features.csv")


Starting 1-minute aggregation... (this may take a bit)

✅ Aggregation complete.
Shape: (35482, 72)
Time range: 2025-10-16 20:13:00 → 2025-11-10 11:34:00

--- 5 first rows ---


,ts,network.bytes_count,network.bytes_sum,network.bytes_mean,network.bytes_std,network.bytes_max,network.bytes_min,network.packets_count,network.packets_sum,network.packets_mean,...,zeek.connection.missed_bytes_sum,zeek.connection.missed_bytes_mean,zeek.connection.missed_bytes_std,zeek.connection.missed_bytes_max,zeek.connection.missed_bytes_min,unique_src_ips,unique_dst_ips,unique_dst_ports,bytes_per_flow_mean,packets_per_flow_mean
0,2025-10-16 20:13:00,158,602578,3813.784810,4208.648624,27234.0,52.0,158,8684,54.962025,...,64,0.405063,3.361524,38.0,0.0,14,11,8,3813.784810,54.962025
1,2025-10-16 20:14:00,1032,5016191,4860.650194,7153.627060,176298.0,104.0,1032,73561,71.280039,...,11854,11.486434,161.203421,4244.0,0.0,27,35,15,4860.650194,71.280039
2,2025-10-16 20:15:00,1277,4638701,3632.498825,4656.860518,59426.0,40.0,1277,69527,54.445576,...,16757,13.122161,303.783755,9087.0,0.0,29,44,18,3632.498825,54.445576
3,2025-10-16 20:16:00,1220,4908569,4023.417213,4760.170356,57596.0,40.0,1220,74649,61.187705,...,10206,8.365574,181.349813,5931.0,0.0,28,38,16,4023.417213,61.187705
4,2025-10-16 20:17:00,899,3957895,4402.552836,4892.929988,59452.0,104.0,899,55105,61.295884,...,13524,15.043382,282.668542,7334.0,0.0,27,41,16,4402.552836,61.295884



--- 5 last rows ---


,ts,network.bytes_count,network.bytes_sum,network.bytes_mean,network.bytes_std,network.bytes_max,network.bytes_min,network.packets_count,network.packets_sum,network.packets_mean,...,zeek.connection.missed_bytes_sum,zeek.connection.missed_bytes_mean,zeek.connection.missed_bytes_std,zeek.connection.missed_bytes_max,zeek.connection.missed_bytes_min,unique_src_ips,unique_dst_ips,unique_dst_ports,bytes_per_flow_mean,packets_per_flow_mean
35477,2025-11-10 11:30:00,524,2352303,4489.127863,9626.728704,62404.0,80.0,524,11930,22.767176,...,12100,23.091603,319.541425,6548.0,0.0,47,70,26,4489.127863,22.767176
35478,2025-11-10 11:31:00,547,3244295,5931.069470,18014.310943,244424.0,80.0,547,12582,23.001828,...,4516,8.255941,191.681255,4483.0,0.0,45,100,27,5931.069470,23.001828
35479,2025-11-10 11:32:00,497,2509731,5049.760563,11901.717919,124336.0,56.0,497,9823,19.764588,...,14349,28.871227,428.637817,8362.0,0.0,47,89,31,5049.760563,19.764588
35480,2025-11-10 11:33:00,545,2098102,3849.728440,16379.781702,285098.0,80.0,545,10377,19.040367,...,1460,2.678899,62.539529,1460.0,0.0,43,65,22,3849.728440,19.040367
35481,2025-11-10 11:34:00,464,3882359,8367.153017,31260.116239,310752.0,80.0,464,11312,24.379310,...,21948,47.301724,652.905184,12978.0,0.0,19,44,19,8367.153017,24.379310



Columns in aggregated dataset:
['ts', 'network.bytes_count', 'network.bytes_sum', 'network.bytes_mean', 'network.bytes_std', 'network.bytes_max', 'network.bytes_min', 'network.packets_count', 'network.packets_sum', 'network.packets_mean', 'network.packets_std', 'network.packets_max', 'network.packets_min', 'duration_s_count', 'duration_s_sum', 'duration_s_mean', 'duration_s_std', 'duration_s_max', 'duration_s_min', 'bps_count', 'bps_sum', 'bps_mean', 'bps_std', 'bps_max', 'bps_min', 'pps_count', 'pps_sum', 'pps_mean', 'pps_std', 'pps_max', 'pps_min', 'bpp_count', 'bpp_sum', 'bpp_mean', 'bpp_std', 'bpp_max', 'bpp_min', 'src_to_dst_byte_ratio_count', 'src_to_dst_byte_ratio_sum', 'src_to_dst_byte_ratio_mean', 'src_to_dst_byte_ratio_std', 'src_to_dst_byte_ratio_max', 'src_to_dst_byte_ratio_min', 'dst_to_src_byte_ratio_count', 'dst_to_src_byte_ratio_sum', 'dst_to_src_byte_ratio_mean', 'dst_to_src_byte_ratio_std', 'dst_to_src_byte_ratio_max', 'dst_to_src_byte_ratio_min', 'src_to_dst_pkt_rat

In [8]:
# =======================
# 🔹 Cell 4 — Per-Feature Anomaly Tagging (No IF/AE)
# =======================
import numpy as np
import pandas as pd

# --- تنظیمات مورد نظر تو ---
PERFEAT_NONZERO_FRAC = 0.00003   # 0.003% of rows per feature → sev>0
PERFEAT_SEV99_FRAC   = 0.000003  # 0.0003% of rows per feature → sev=99

# لیست فیچرهایی که می‌خوای بررسی بشن (همون‌هایی که گفتی)
ROW_FEATURES = [
    "network.protocol", "network.transport",
    "zeek.connection.missed_bytes",
    "destination.ip", "destination.port", "destination.packets",
    "source.ip", "source.packets",
    "network.packets", "network.bytes",
]

# روش امتیازدهی هر فیچر: "rarity" برای گسسته‌ها؛ "high_tail" برای عددی‌ها
SCORING_MODE = {
    "network.protocol": "rarity",
    "network.transport": "rarity",
    "zeek.connection.missed_bytes": "high_tail",
    "destination.ip": "rarity",
    "destination.port": "rarity",
    "destination.packets": "high_tail",
    "source.ip": "rarity",
    "source.packets": "high_tail",
    "network.packets": "high_tail",
    "network.bytes": "high_tail",
}

# --- توابع کمکی ---
def map_scores_to_severity(scores, nonzero_frac, sev99_frac):
    """Map a 1D score array to 0..99 severity by top-frac thresholds (feature-wise)."""
    x = np.asarray(scores, dtype=np.float64)
    # رتبه درصدی (ascending). چون امتیاز بالا = غیرعادی، از خود x استفاده می‌کنیم.
    order = np.argsort(x)
    ranks = np.empty_like(order, dtype=np.float64)
    ranks[order] = np.arange(len(x))
    pr = ranks / max(len(x) - 1, 1)

    thr_nonzero = 1.0 - nonzero_frac
    thr_99 = 1.0 - sev99_frac

    sev = np.zeros_like(x, dtype=np.int16)
    sev[pr >= thr_99] = 99
    mid = (pr >= thr_nonzero) & (pr < thr_99)
    if mid.any():
        pr_mid = (pr[mid] - thr_nonzero) / max(thr_99 - thr_nonzero, 1e-12)
        sev_mid = 1 + np.floor(pr_mid * 98).astype(np.int16)
        sev[mid] = np.clip(sev_mid, 1, 98)
    return sev

def rarity_score(series: pd.Series) -> np.ndarray:
    """Rarity = -log10(freq/N). مقدار بزرگ‌تر ⇒ نایاب‌تر ⇒ مشکوک‌تر."""
    s = series.astype("object")
    vc = s.value_counts(dropna=False)
    freq = s.map(vc).astype(np.float64)
    N = float(len(s))
    return -np.log10(np.maximum(freq / N, 1.0 / (N * 1e6)))  # clamp برای صفر نشدن

def high_tail_score(series: pd.Series) -> np.ndarray:
    """High-tail anomaly score برای عددی‌ها: z-score مثبت (منفی‌ها صفر)."""
    x = pd.to_numeric(series, errors="coerce").astype("float64").fillna(0.0).values
    mu = np.mean(x); sd = np.std(x) + 1e-12
    z = (x - mu) / sd
    z[z < 0] = 0.0       # فقط دُم بالایی
    return z

def ensure_columns(df, cols):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise SystemExit(f"❌ Missing required columns: {missing}")

# --- ورودی مورد نیاز: df_clean از سلول 2
try:
    _ = df_clean.shape
except Exception:
    raise SystemExit("df_clean is required. Please run Cell 2 first.")

# فقط ستون‌های مورد نیاز را برمی‌داریم (به ترتیب مشخص)
ensure_columns(df_clean, ROW_FEATURES + ["ts"])
base_df = df_clean[ROW_FEATURES + ["ts"]].copy()

# امتیاز و severity فیچر-به-فیچر
feat_scores = {}
feat_sev    = {}

for feat in ROW_FEATURES:
    mode = SCORING_MODE.get(feat, "high_tail")
    if mode == "rarity":
        scores = rarity_score(df_clean[feat])
    else:
        scores = high_tail_score(df_clean[feat])

    sev = map_scores_to_severity(scores, PERFEAT_NONZERO_FRAC, PERFEAT_SEV99_FRAC)
    feat_scores[feat] = scores
    feat_sev[feat] = sev

# DataFrame خروجی ردیفی (فقط فیچرهای خواسته‌شده)
row_pf = base_df.copy()

# الحاق severity هر فیچر (ستون‌های: {feat}_sev و {feat}_score)
for feat in ROW_FEATURES:
    row_pf[f"{feat}__score"] = feat_scores[feat]
    row_pf[f"{feat}__sev"]   = feat_sev[feat].astype(np.int16)

# انومالی نهایی هر ردیف = مجموع severity فیچرها
sev_cols = [f"{f}__sev" for f in ROW_FEATURES]
row_pf["row_final_sev_sum"]   = row_pf[sev_cols].sum(axis=1).astype(np.int32)
row_pf["row_final_sev_capped"] = np.minimum(row_pf["row_final_sev_sum"], 99).astype(np.int16)  # اگر سقف 99 می‌خوای

# ساخت ستون «کدام فیچرها مشکوک بودن»
def join_anoms(row):
    names = [f for f in ROW_FEATURES if row[f"{f}__sev"] > 0]
    return ";".join(names)

row_pf["row_suspicious_features"] = row_pf.apply(join_anoms, axis=1)

# گزارش کوچک
nz_share = (row_pf[sev_cols] > 0).any(axis=1).mean()
print(f"[Per-Feature] Share of rows with ANY anomalous feature (>0): {nz_share:.6f}")
print("Example head:")
print(row_pf.head(3)[["ts","row_final_sev_sum","row_final_sev_capped","row_suspicious_features"]])
# ذخیره موقت برای سلول 5
row_pf.to_csv("row_perfeature_scores.csv", index=False)
print("\nSaved: row_perfeature_scores.csv")


[Per-Feature] Share of rows with ANY anomalous feature (>0): 0.000203
Example head:
                       ts  row_final_sev_sum  row_final_sev_capped  \
0 2025-10-16 20:13:49.580                  0                     0   
1 2025-10-16 20:13:49.681                  0                     0   
2 2025-10-16 20:13:49.691                  0                     0   

  row_suspicious_features  
0                          
1                          
2                          

Saved: row_perfeature_scores.csv


In [9]:
# =======================
# 🔹 Cell 5 — Export & Download (Per-Feature)
# =======================
import os, zipfile
import pandas as pd
from google.colab import files

# Load
if not os.path.exists("row_perfeature_scores.csv"):
    raise SystemExit("❌ Run Cell 4 first.")
row_pf = pd.read_csv("row_perfeature_scores.csv", low_memory=False)

# فقط ردیف‌هایی که حداقل یک فیچر sev>0 دارند
sev_cols = [c for c in row_pf.columns if c.endswith("__sev")]
row_anoms = row_pf[(row_pf[sev_cols] > 0).any(axis=1)].copy()

# مرتب‌سازی: اول مجموع severity بیشتر، بعد capped
row_anoms = row_anoms.sort_values(
    ["row_final_sev_sum", "row_final_sev_capped"],
    ascending=[False, False]
)

# ذخیره CSV و Excel
row_csv = "row_perfeature_anomalies_gt0.csv"
row_anoms.to_csv(row_csv, index=False)

# Excel
try:
    import xlsxwriter  # noqa
except Exception:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "xlsxwriter", "-q"], check=True)

xlsx = "row_perfeature_anomalies_gt0.xlsx"
with pd.ExcelWriter(xlsx, engine="xlsxwriter") as xlw:
    row_anoms.to_excel(xlw, sheet_name="rows_sev_gt0_perfeature", index=False)

# Zip
bundle = "row_perfeature_anomalies_gt0_bundle.zip"
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for f in [row_csv, xlsx]:
        if os.path.exists(f):
            zf.write(f)

print("✅ Files ready:")
print(" -", row_csv)
print(" -", xlsx)
print(" -", bundle)

files.download(bundle)


✅ Files ready:
 - row_perfeature_anomalies_gt0.csv
 - row_perfeature_anomalies_gt0.xlsx
 - row_perfeature_anomalies_gt0_bundle.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>